![OneHealth DataSpace](https://bigdata.dataspace.cesga.es/static/images/public/imagotipo.png)

> **⚠️ ADVERTENCIA**: actualmente la plataforma está en fase de pueba. Agradecemos que nos hagáis llegar cualquier comentario a onehealth@cesga.es. 

# Exportación a csv y descarga de los datos avanzada
En este **caso de uso** vamos a explorar distintas opciones de exportación de los datos a archivos `csv` y cómo descargarlos en nuestro equipo.

In [15]:
import onehealth
import pyspark.sql.functions as F
from pyspark.sql.functions import col, expr, min, max, to_date
import pandas as pd

In [16]:
afloramiento = onehealth.load("afloramento/intecmar_calculo")

Pero no necesitamos almacenar todas las filas del conjunto de datos; podemos, por ejemplo, consultar las primeras 10 filas utilizando la función `limit`, después aplicamos la función `toPandas` para transformar los datos. El notebook mostrará la tabla correspondiente al no indicar ninguna acción adicional.

Para almacenar estos datos en un archivo CSV en nuestro directorio, crearemos una variable donde almacenaremos el conjunto de datos filtrado. Después utilizaremos la función `to_csv`:
- Si no indicamos nada, guardará el archivo en la misma carpeta donde está almacenado el notebook.

In [17]:
# Almacenamos el dataset en un DataFrame de Pandas
descargar = afloramiento.limit(10).toPandas()

# Convertimos y guardamos el nuevo DataFrame
descargar.to_csv('~/notebook/descargar_csv/afloramiento.csv')

OSError: Cannot save file into a non-existent directory: '/home/cesga/dmartinez/notebook/descargar_csv'

Al entrar en la carpeta <b>"descargar_csv"</b> podemos abrir directamente el archivo y se mostrará en una nueva pestaña. La exportación por defecto utiliza la coma "," como separador.

### Exportar sin cabecera y sin índices

Por defecto la función `to_csv` exporta los datos incluyendo las cabeceras, es decir, los nombres de las columnas. Para exportar el archivo CSV sin las cabeceras de las columnas utilizamos el parámetro <b>"header=False"</b>.

In [18]:
descargar.to_csv('~/notebook/descargar_csv/afloramiento_sin_cabecera.csv', header=False)

OSError: Cannot save file into a non-existent directory: '/home/cesga/dmartinez/notebook/descargar_csv'

También, por defecto, incluye un índice de fila; en este caso, un número que identifica cada fila. Con el parámetro <b>"index=False"</b> podemos prescindir de esa columna por completo en nuestro archivo CSV. Por ejemplo, un archivo sin cabeceras ni índice se generaría utilizando los siguientes parámetros:

In [7]:
descargar.to_csv('~/notebook/descargar_csv/afloramiento_sin_cabecera_sin_indice.csv', header=False, index=False)

### Exportar a un csv comprimiendo el archivo

La función `to_csv` acepta otros parámetros, como la compresión de archivos utilizando formatos como `zip`, `gzip`, `bz2`, `zstd`, `xz` y `tar`, para ello, tenemos que indicarlo mediante un diccionario.

Como ejemplo:
- Creamos una variable llamada <b>opc_compresion</b> que será nuestro diccionario con las opciones necesarias para la compresión. En este ejemplo, el diccionario indica el método de compresión <b>(method='zip')</b> y el nombre del archivo CSV que se va a comprimir <b>(archive_name='afloramiento.csv')</b>. Este archivo se genera automáticamente mediante la función `to_csv`.
- Utilizamos el diccionario como parámetro de la función `to_csv`.

In [8]:
# Creamos el diccionario
opc_compresion = dict(method='zip',
                     archive_name='afloramiento_filtro_filas.csv')

# Usamos el diccionario
descargar.to_csv('~/notebook/descargar_csv/afloramiento.zip', compression=opc_compresion)

### Exportar a un csv usando un separador distinto

El parámetro <b>'sep'</b> permite elegir un separador distinto, ya que por defecto es la coma `,`.

In [9]:
descargar.to_csv('~/notebook/descargar_csv/afloramiento_separador_punto_y_coma.csv',sep=";")

### Utilizando varias opciones simultáneamente

Podemos, por ejemplo, generar un archivo comprimido que contenga un CSV que utilice `;` como separador y que no tenga cabeceras.

In [10]:
# Creamos el diccionario
opc_compresion = dict(method='zip',
                     archive_name='afloramiento_separador_sincabecera.csv')

# Usamos el diccionario
descargar.to_csv('~/notebook/descargar_csv/afloramiento_separador_sincabecera.zip', header=False, sep=";", compression=opc_compresion)

### Crear un archivo CSV después de filtrar los datos

Podemos utilizar las opciones de Pandas para filtrar nuestros datos y almacenar el resultado de dichos filtros en un archivo CSV. En este caso, tenemos varias condiciones en nuestro filtro:
- `water_u` debe tener un valor y este debe ser mayor que 0,0. Como ejemplo, seleccionamos la columna que se va a filtrar utilizando el formato `nombre_dataframe.nombre_columna`.
- La latitud y la longitud deben coincidir con las indicadas. Como ejemplo, seleccionamos la columna que se va a filtrar utilizando la función `col(nombre_columna)`.

Filtramos el conjunto de datos buscando aquellas filas que contengan algún valor en la columna `water_u` y que tengan una latitud concreta:

In [11]:
filtrado = afloramiento.filter((afloramiento.water_u!='none') & (afloramiento.water_u>0.0) & (col('latitude')==40.78689956665039) & (col('longitude')==-9.502409934997559))

Podemos seguir aplicando filtros cuando sea necesario. Por ejemplo, en este caso solo mostramos las filas con `site_status==1`.

In [12]:
filtrado = filtrado.filter(col('site_status')==1)

De esos datos ya filtrados vamos a seleccionar un conjunto de columnas:

In [13]:
columnas = filtrado [["time","latitude","longitude","water_u"]]

# Pasamos as primeiras 100 filas a Pandas
columnas = columnas.limit(100).toPandas()

Almacenamos este resultado en un archivo CSV utilizando lo ya aprendido:

In [14]:
columnas.to_csv('~/notebook/descargar_csv/afloramiento_filtrado.csv', index=False, sep=";")

Al entrar en la carpeta <b>"descargar_csv"</b> podemos abrir directamente el archivo y se mostrará en una nueva pestaña.

### Descarga de los archivos en nuestro equipo

Para descargar los archivos generados mediante la exportación a CSV (ya sean archivos CSV o comprimidos), solamente tenemos que hacer clic con el botón derecho del ratón sobre el archivo en el panel de navegación situado a la izquierda y seleccionar "Download". También podemos descargar varios archivos a la vez si los seleccionamos todos.

![OneHealth DataSpace](../../../static/como-descargar-datos-pc.jpg)